# 10 — Arabic Language Support Demo

This notebook demonstrates end-to-end Arabic text support:
preprocessing, tokenization, generation, and RTL rendering.

## 1. Arabic Text Preprocessing

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.data.preprocessor import DataPreprocessor

dp = DataPreprocessor(normalize_arabic=True, alef_normalization=True)

# Demonstrate alef normalization
raw = '\u0623\u0647\u0644\u0627 \u0625\u0628\u0631\u0627\u0647\u064a\u0645 \u0622\u0645\u0646\u0629'  # أهلا إبراهيم آمنة
import re
_ALEF_VARIANTS = re.compile('[\u0622\u0623\u0625]')
normalized = _ALEF_VARIANTS.sub('\u0627', raw)
print(f'Raw:        {raw}')
print(f'Normalized: {normalized}')

## 2. Loading Arabic SFT Datasets

In [ ]:
stories = dp.parse_sft_data('../data/finetune/story_completion/arabic_stories.jsonl')
poetry = dp.parse_sft_data('../data/finetune/poetry/arabic_poetry.jsonl')
general = dp.parse_sft_data('../data/finetune/arabic_sft.jsonl')

print(f'Story completion records: {len(stories)}')
print(f'Poetry records:           {len(poetry)}')
print(f'General Arabic SFT:       {len(general)}')
print(f'Total:                    {len(stories) + len(poetry) + len(general)}')

In [ ]:
# Show a sample story record
sample = stories[0]
print('Instruction:', sample['instruction'])
if 'input' in sample:
    print('Input:      ', sample['input'])
print('Output:     ', sample['output'][:100], '...')

## 3. Arabic Tokenization with BPE

In [ ]:
from src.tokenizer.bpe_tokenizer import BPETokenizer

# Build corpus from all Arabic data
all_records = stories + poetry + general
corpus = ' '.join(
    r['instruction'] + ' ' + r.get('input', '') + ' ' + r['output']
    for r in all_records
)
print(f'Corpus size: {len(corpus)} chars, {len(corpus.encode("utf-8"))} bytes')

In [ ]:
tok = BPETokenizer(vocab_size=500)
tok.train(corpus)
print(f'Vocabulary size: {len(tok.vocab)}')

In [ ]:
# Encode/decode Arabic text
arabic_text = 'مرحبا بالعالم العربي'
ids = tok.encode(arabic_text)
decoded = tok.decode(ids)
print(f'Original:  {arabic_text}')
print(f'Token IDs: {ids}')
print(f'Decoded:   {decoded}')
print(f'Round-trip: {decoded == arabic_text}')

In [ ]:
# Mixed Arabic-English text
mixed = 'GPT model يدعم اللغة العربية'
ids_m = tok.encode(mixed)
decoded_m = tok.decode(ids_m)
print(f'Original:  {mixed}')
print(f'Decoded:   {decoded_m}')
print(f'Round-trip: {decoded_m == mixed}')

## 4. Arabic Text Generation (Untrained Model)

In [ ]:
import torch
from src.config import GPTConfig
from src.model.transformer import GPTModel
from src.generation.generator import TextGenerator

config = GPTConfig(vocab_size=len(tok.vocab), d_model=64, num_heads=4,
                   num_layers=2, max_seq_len=128, dropout_rate=0.1)
config.validate()
model = GPTModel(config)
model.eval()
gen = TextGenerator(model, tok)
print(f'Model parameters: {model.count_parameters():,}')

In [ ]:
# Generate from Arabic prompt (random output since untrained)
prompt = 'في يوم من الأيام'
text, ids = gen.generate(prompt, max_new_tokens=30, temperature=0.8)
print(f'Prompt: {prompt}')
print(f'Generated: {text}')

## 5. RTL Rendering for Arabic Text

In [ ]:
from IPython.display import HTML, display

def render_rtl(text, label=''):
    html = f'''
    <div style="direction: rtl; text-align: right; font-family: 'Noto Sans Arabic', Arial;
                padding: 12px; margin: 8px 0; background: #f8f9fa; border-radius: 6px;
                border-right: 4px solid #2196F3; font-size: 16px; line-height: 1.8;">
        {f"<strong>{label}</strong><br>" if label else ""}
        {text}
    </div>'''
    display(HTML(html))

# Render Arabic samples
render_rtl(stories[0]['output'], 'قصة')
render_rtl(poetry[0]['output'], 'شعر')

In [ ]:
# Render multiple poetry samples with RTL
for i, rec in enumerate(poetry[:3]):
    render_rtl(rec['output'], rec['instruction'])

## Summary

This notebook demonstrated:
- Arabic alef normalization (آ أ إ → ا)
- Loading Arabic SFT datasets (stories, poetry, general)
- Byte-level BPE tokenization with Arabic round-trip
- Text generation with Arabic prompts
- RTL rendering for Arabic text display